# 5G Network Intelligence — Fixed Blend Candidate Audit

## Development evidence and question

The blend chosen on inner1 reduced inner2 MAE 0.318% / 0.556% / 0.216% at 1/6/24h. It is fixed at 20% B for 1/6h and 20% A for 24h. Test whether the effect persists on reused validation; do not change these coefficients.

## Frozen audit procedure

Refit A/B through target hour 503 for the rounded median of development best_epoch+1 (A=4, B=3). No early stopping. Reuse the previously verified hurdle validation forecasts unchanged. Only after refitting, audit 504–671 on all original eligible beams. No hyperparameter search, calibration, final holdout replay or production promotion.


In [ ]:
RUN_TRAINING = False
SOURCE_COMMIT = 'c31f04f8d8fdbeeceabb32cfd45ba63eba640d28'


## Fixed protocol and pinned evidence

The saved selection and hurdle replay must match their pinned SHA256 values. The original hurdle is not retrained.


In [ ]:
AUDIT_PROTOCOL = {'protocol': 'fixed-hurdle-linear-validation-audit-v1',
 'fit_end': 503,
 'audit_end': 671,
 'horizons': [1, 6, 24],
 'epoch_policy': 'rounded_median_of_two_development_best_epoch_plus_one',
 'selection_sha256': 'a2a586bee68fe7797b9f21313acee5c6e930ec67c8071a06ac8f61b925e6fd07',
 'hurdle_replay_sha256': 'ec7bc06e9c6d205276436328f64117ce3b7c0fc994f08dcca907a44893e41dc2',
 'early_stopping': 'forbidden',
 'coefficient_changes': 'forbidden',
 'hurdle_retraining': 'forbidden',
 'original_final_holdout': 'forbidden',
 'production_change': 'forbidden'}


## Completed hosted source inputs

Attach Temporal Linear, development blend and the original zero-inflated candidate audit outputs. A small MIT source attachment supplies the new fixed-epoch trainer and forecast arithmetic. No repository source dump is embedded.


In [ ]:
from pathlib import Path
import os,sys,shutil,subprocess,tarfile
if RUN_TRAINING:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Hosted Kaggle required')
    linear_roots=[p.parent.parent for p in Path('/kaggle/input').rglob('temporal_linear_development.csv') if (p.parent.parent/'artifacts/temporal_linear/manifest.json').exists()]
    blend_roots=[p.parent.parent for p in Path('/kaggle/input').rglob('development_blend_confirmation.csv') if (p.parent.parent/'artifacts/development_blend/manifest.json').exists()]
    hurdle_replays=list(Path('/kaggle/input').rglob('hurdle_candidate_val.npz'))
    if len(linear_roots)!=1 or len(blend_roots)!=1 or len(hurdle_replays)!=1:
        raise RuntimeError('Attach exactly one completed output for each required source')
    LINEAR=linear_roots[0]/'artifacts/temporal_linear'
    BLEND=blend_roots[0]/'artifacts/development_blend'
    HURDLE_REPLAY=hurdle_replays[0]
    PROJECT=Path('/kaggle/working/blend_candidate')
    PROJECT.mkdir(exist_ok=True)
    for folder in ('src','scripts','configs','data/processed'):
        shutil.copytree(linear_roots[0]/folder,PROJECT/folder,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
    os.chdir(PROJECT)
    sys.path.insert(0,str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT']=SOURCE_COMMIT
    Path('reports').mkdir(exist_ok=True)
    patches=list(Path('/kaggle/input').rglob('finalize_blend_candidate.py'))
    if len(patches)!=1:
        raise RuntimeError('Attach exactly one fixed-audit source')
    patch=patches[0].parent
    for name,folder in (('finalize_blend_candidate.py','scripts'),('forecast_blend.py','src/models'),('blend_candidate_audit.yaml','configs')):
        shutil.copyfile(patch/name,Path(folder)/name)


## GPU and fitting controls

Use the hosted image's PyTorch. Missing CUDA stops fitting. Save every epoch's final state and training loss; do not choose a best epoch from validation. The last fixed epoch is the candidate.


In [ ]:
if RUN_TRAINING:
    import torch
    print('PyTorch',torch.__version__)
    if not torch.cuda.is_available():
        raise RuntimeError('Hosted CUDA GPU required')
    subprocess.run(['nvidia-smi'],check=True)


## Fixed refit and audit

Two neural fits, seven epochs in total. After fitting, construct the declared mixture and retain all component/audit scores. A validation gain is exploratory evidence, not an unseen-test guarantee.


In [ ]:
if RUN_TRAINING:
    command=[sys.executable,'-u','-m','scripts.finalize_blend_candidate','--linear-root',str(LINEAR),'--blend-root',str(BLEND),'--hurdle-replay',str(HURDLE_REPLAY)]
    with Path('reports/blend_candidate_training.log').open('w') as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        status=process.wait()
    with tarfile.open('/kaggle/working/blend_candidate_results.tar.gz','w:gz') as archive:
        for folder in ('reports','artifacts/blend_candidate'):
            if Path(folder).exists():
                archive.add(folder,arcname=folder)
    if status:
        raise RuntimeError('Fixed audit failed; inspect saved logs/checkpoints')
else:
    print('Review mode: no fitting or calibration.')


## Saved metrics and limitations

MAE uses original units, WAPE is a ratio. Report every horizon and full original population. Keep negative results, inspect zero false-positive rates and independently verify checkpoint/forecast consistency after download. Repeated validation cannot prove no overfitting or future performance. Production is unchanged.


In [ ]:
paths=[Path('reports/blend_candidate_validation.csv'),Path('../reports/blend_candidate_validation.csv')]
if Path('/kaggle/input').is_dir():
    paths+=list(Path('/kaggle/input').rglob('blend_candidate_validation.csv'))
path=next((p for p in paths if p.exists()),None)
if path:
    import pandas as pd
    from IPython.display import display
    frame=pd.read_csv(path)
    display(frame.loc[frame.load_tier=='All',['model','horizon','mae','wape','n_valid']])
else:
    print('No completed audit scores attached.')


## Completed v1 audit and trade-offs

Fixed blend MAE was 0.226834 / 0.250427 / 0.249162 at 1/6/24h. WAPE was 40.86% / 44.42% / 44.94%. This improves hurdle at 1/6h but slightly regresses at 24h. The 24h exploratory base-station interval crosses zero; NR3500 transfer retains lower 6/24h point MAE. Zero false-positive rates also worsen. All ten artifact files, full blend arithmetic and 81 history-only component predictions were verified. Keep the declared weights, record the regression, and do not promote from reused validation. The 10% WAPE target remains unmet.
